<a href="https://colab.research.google.com/github/ladams1204/SportsBookEdge/blob/main/07_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install streamlit pyngrok --quiet

# Check installs
!streamlit --version
print("Ready")

Streamlit, version 1.56.0
Ready


In [17]:
!pip install nba_api streamlit pyngrok --quiet
print("Installed")

Installed


In [5]:
# Recreate the database in this notebook's runtime
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players
import sqlite3
import pandas as pd
import time

conn = sqlite3.connect('sportsbook.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS nba_player_games (
    game_id TEXT, player_id INTEGER, player_name TEXT, game_date TEXT,
    matchup TEXT, wl TEXT, minutes INTEGER, points INTEGER, rebounds INTEGER,
    assists INTEGER, steals INTEGER, blocks INTEGER, turnovers INTEGER,
    fgm INTEGER, fga INTEGER, fg3m INTEGER, fg3a INTEGER, plus_minus INTEGER,
    season TEXT, PRIMARY KEY (game_id, player_id)
)
''')
conn.commit()

target_players = [
    "LeBron James", "Stephen Curry", "Nikola Jokic", "Luka Doncic",
    "Jayson Tatum", "Giannis Antetokounmpo", "Shai Gilgeous-Alexander",
    "Kevin Durant", "Anthony Edwards", "Joel Embiid",
    "Devin Booker", "Damian Lillard", "Jimmy Butler", "Paul George",
    "Donovan Mitchell", "Trae Young", "De'Aaron Fox", "Ja Morant",
    "Anthony Davis", "Victor Wembanyama",
    "Alperen Sengun", "Tyrese Maxey", "Jaylen Brown", "Cade Cunningham",
    "Paolo Banchero", "Franz Wagner", "Amen Thompson", "Jalen Green",
    "Chet Holmgren", "Jalen Williams"
]

def pull(name, season):
    matches = players.find_players_by_full_name(name)
    if not matches:
        return
    pid = matches[0]['id']
    log = playergamelog.PlayerGameLog(player_id=pid, season=season)
    df = log.get_data_frames()[0]
    if len(df) == 0:
        return
    df['SEASON'] = season
    df['PLAYER_NAME'] = matches[0]['full_name']
    df = df.rename(columns={
        'Game_ID': 'game_id', 'Player_ID': 'player_id',
        'PLAYER_NAME': 'player_name', 'GAME_DATE': 'game_date',
        'MATCHUP': 'matchup', 'WL': 'wl', 'MIN': 'minutes',
        'PTS': 'points', 'REB': 'rebounds', 'AST': 'assists',
        'STL': 'steals', 'BLK': 'blocks', 'TOV': 'turnovers',
        'FGM': 'fgm', 'FGA': 'fga', 'FG3M': 'fg3m', 'FG3A': 'fg3a',
        'PLUS_MINUS': 'plus_minus', 'SEASON': 'season'
    })[['game_id', 'player_id', 'player_name', 'game_date', 'matchup', 'wl',
        'minutes', 'points', 'rebounds', 'assists', 'steals', 'blocks',
        'turnovers', 'fgm', 'fga', 'fg3m', 'fg3a', 'plus_minus', 'season']]
    for _, row in df.iterrows():
        try:
            cursor.execute(f"INSERT INTO nba_player_games VALUES ({','.join(['?']*19)})", tuple(row))
        except sqlite3.IntegrityError:
            pass
    conn.commit()

print("Pulling data (~3 min)...")
for season in ['2024-25', '2025-26']:
    for name in target_players:
        pull(name, season)
        time.sleep(0.6)

count = cursor.execute("SELECT COUNT(*) FROM nba_player_games").fetchone()[0]
print(f"Database ready: {count} rows")
conn.close()

Pulling data (~3 min)...
Database ready: 3364 rows


In [21]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime

st.set_page_config(
    page_title="SportsBook Edge",
    page_icon="🏀",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    .main { padding-top: 1rem; }
    [data-testid="stMetric"] {
        background-color: #1a1d23;
        padding: 15px;
        border-radius: 8px;
        border: 1px solid #2d3139;
    }
    [data-testid="stMetricLabel"] { color: #8b92a0; font-size: 0.8rem; }
    [data-testid="stMetricValue"] { color: #e8eaed; font-size: 1.8rem; }
    h1 { color: #e8eaed; }
    h2, h3 { color: #c9cdd4; }
</style>
""", unsafe_allow_html=True)

@st.cache_data
def load_games():
    conn = sqlite3.connect('sportsbook.db')
    df = pd.read_sql("SELECT * FROM nba_player_games ORDER BY player_id, game_date", conn)
    df['game_date'] = pd.to_datetime(df['game_date'])
    for col in ['points', 'rebounds', 'assists', 'minutes', 'fgm', 'fga', 'fg3m', 'fg3a']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    conn.close()
    return df

@st.cache_data
def load_picks():
    try:
        df = pd.read_csv('multi_market_picks.csv')
        return df
    except FileNotFoundError:
        return pd.DataFrame()

games_df = load_games()
picks_df = load_picks()

st.sidebar.title("🏀 SportsBook Edge")
st.sidebar.markdown("---")
page = st.sidebar.radio(
    "Navigate",
    ["Overview", "Today's Picks", "Player Explorer", "Performance Tracker"]
)
st.sidebar.markdown("---")
st.sidebar.caption(f"Data as of {datetime.now().strftime('%b %d, %Y')}")
st.sidebar.caption(f"{len(games_df)} games | {games_df['player_name'].nunique()} players")

if page == "Overview":
    st.title("📊 SportsBook Edge Dashboard")
    st.markdown("Personal NBA prop betting analytics")
    st.markdown("---")

    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Total Picks Tracked", len(picks_df))
    with col2:
        st.metric("Players in Database", games_df['player_name'].nunique())
    with col3:
        st.metric("Games Analyzed", len(games_df))
    with col4:
        if len(picks_df) > 0 and 'market' in picks_df.columns:
            st.metric("Markets Covered", picks_df['market'].nunique())
        else:
            st.metric("Markets Covered", "—")

    st.markdown("---")
    st.subheader("Points Distribution (All Tracked Players)")
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=games_df['points'].dropna(),
        nbinsx=40,
        marker_color='#4a9eff',
        opacity=0.8
    ))
    fig.update_layout(
        template='plotly_dark',
        height=400,
        xaxis_title="Points Scored",
        yaxis_title="Frequency",
        showlegend=False
    )
    st.plotly_chart(fig, use_container_width=True)

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Top 10 Scoring Averages")
        top_scorers = (games_df.groupby('player_name')['points'].mean()
                       .sort_values(ascending=False).head(10).round(1).reset_index())
        top_scorers.columns = ['Player', 'Avg Points']
        st.dataframe(top_scorers, use_container_width=True, hide_index=True)
    with col2:
        st.subheader("Top 10 Rebounders")
        top_reb = (games_df.groupby('player_name')['rebounds'].mean()
                   .sort_values(ascending=False).head(10).round(1).reset_index())
        top_reb.columns = ['Player', 'Avg Rebounds']
        st.dataframe(top_reb, use_container_width=True, hide_index=True)

elif page == "Today's Picks":
    st.title("🎯 Current Picks")
    if len(picks_df) == 0:
        st.warning("No picks tracked yet. Run Session 6 to generate picks.")
    else:
        st.markdown(f"**{len(picks_df)} picks in tracking log**")
        col1, col2 = st.columns(2)
        with col1:
            markets = ['All'] + sorted(picks_df['market'].unique().tolist()) if 'market' in picks_df else ['All']
            selected_market = st.selectbox("Market", markets)
        with col2:
            leans = ['All'] + sorted(picks_df['lean'].unique().tolist()) if 'lean' in picks_df else ['All']
            selected_lean = st.selectbox("Lean", leans)

        filtered = picks_df.copy()
        if selected_market != 'All' and 'market' in filtered:
            filtered = filtered[filtered['market'] == selected_market]
        if selected_lean != 'All' and 'lean' in filtered:
            filtered = filtered[filtered['lean'] == selected_lean]

        display_cols = ['player_name', 'game', 'market', 'our_prediction',
                        'consensus_line', 'diff_vs_line', 'lean']
        display_cols = [c for c in display_cols if c in filtered.columns]
        st.dataframe(filtered[display_cols], use_container_width=True, hide_index=True)

elif page == "Player Explorer":
    st.title("👤 Player Explorer")
    player_list = sorted(games_df['player_name'].unique())
    selected_player = st.selectbox("Select a player", player_list)

    player_df = games_df[games_df['player_name'] == selected_player].copy()
    player_df = player_df.sort_values('game_date')

    recent = player_df.tail(20)
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Last 20 Avg Points", f"{recent['points'].mean():.1f}")
    with col2:
        st.metric("Last 20 Avg Rebounds", f"{recent['rebounds'].mean():.1f}")
    with col3:
        st.metric("Last 20 Avg Assists", f"{recent['assists'].mean():.1f}")
    with col4:
        st.metric("Last 20 Avg Minutes", f"{recent['minutes'].mean():.1f}")

    st.markdown("---")
    st.subheader(f"{selected_player} — Points by Game")
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=player_df['game_date'],
        y=player_df['points'],
        mode='lines+markers',
        name='Points',
        line=dict(color='#4a9eff', width=2),
        marker=dict(size=6)
    ))
    player_df['rolling_10'] = player_df['points'].rolling(10, min_periods=3).mean()
    fig.add_trace(go.Scatter(
        x=player_df['game_date'],
        y=player_df['rolling_10'],
        mode='lines',
        name='10-Game Rolling Avg',
        line=dict(color='#f59e0b', width=3, dash='dot')
    ))
    fig.update_layout(template='plotly_dark', height=400, xaxis_title="Date", yaxis_title="Points")
    st.plotly_chart(fig, use_container_width=True)

    st.subheader("Last 15 Games")
    recent_display = player_df.tail(15)[
        ['game_date', 'matchup', 'minutes', 'points', 'rebounds', 'assists', 'plus_minus']
    ].sort_values('game_date', ascending=False)
    recent_display['game_date'] = recent_display['game_date'].dt.strftime('%b %d, %Y')
    st.dataframe(recent_display, use_container_width=True, hide_index=True)

elif page == "Performance Tracker":
    st.title("📈 Paper-Trade Performance")
    if len(picks_df) == 0:
        st.warning("No picks in tracking log yet.")
    elif 'actual_value' not in picks_df.columns or picks_df['actual_value'].notna().sum() == 0:
        st.info("Fill in actual_value and result columns in multi_market_picks.csv after games, then reload.")
        st.dataframe(picks_df.head(20), use_container_width=True, hide_index=True)
    else:
        completed = picks_df.dropna(subset=['result']).copy()
        total = len(completed)
        wins = (completed['result'] == 'WIN').sum()
        losses = (completed['result'] == 'LOSS').sum()
        hit_rate = wins / (wins + losses) if (wins + losses) > 0 else 0

        col1, col2, col3, col4 = st.columns(4)
        with col1:
            st.metric("Total Completed", total)
        with col2:
            st.metric("Wins", wins)
        with col3:
            st.metric("Losses", losses)
        with col4:
            st.metric("Hit Rate", f"{hit_rate:.1%}")

st.markdown("---")
st.caption("Built with Streamlit | Data from NBA Stats API and The Odds API")

Overwriting app.py


In [24]:
from pyngrok import ngrok, conf
from google.colab import userdata
import subprocess
import time

# Configure ngrok
conf.get_default().auth_token = userdata.get('NGROK_TOKEN').strip()

# Clean up any existing tunnels
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# Start Streamlit in the background
subprocess.Popen(['streamlit', 'run', 'app.py',
                  '--server.port', '8501',
                  '--server.headless', 'true'])

# Wait for it to boot
time.sleep(5)

# Open the public tunnel
public_url = ngrok.connect(8501)
print(f"\n🚀 Your dashboard is live at:\n")
print(f"   {public_url.public_url}")
print(f"\n(URL is active while this notebook runs)")


🚀 Your dashboard is live at:

   https://turmoil-ellipse-stiffly.ngrok-free.dev

(URL is active while this notebook runs)


In [16]:
# Regenerate the multi-market picks file for the dashboard
# This uses your cached data, so costs zero API credits

import pandas as pd
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players
import time
from datetime import datetime

# Players from your Session 6 slate
expanded_players = [
    ('LeBron James', 'Rockets @ Lakers'),
    ('Alperen Sengun', 'Rockets @ Lakers'),
    ('Amen Thompson', 'Rockets @ Lakers'),
    ('Jabari Smith Jr', 'Rockets @ Lakers'),
    ('Rui Hachimura', 'Rockets @ Lakers'),
    ('Luke Kennard', 'Rockets @ Lakers'),
    ('Deandre Ayton', 'Rockets @ Lakers'),
    ('Tyrese Maxey', 'Sixers @ Celtics'),
    ('Jaylen Brown', 'Sixers @ Celtics'),
    ('Jayson Tatum', 'Sixers @ Celtics'),
    ('Kelly Oubre Jr', 'Sixers @ Celtics'),
    ('Payton Pritchard', 'Sixers @ Celtics'),
    ('Nikola Vucevic', 'Sixers @ Celtics'),
    ('Shai Gilgeous-Alexander', 'Suns @ Thunder'),
    ('Devin Booker', 'Suns @ Thunder'),
    ('Jalen Green', 'Suns @ Thunder'),
    ('Jalen Williams', 'Suns @ Thunder'),
    ('Chet Holmgren', 'Suns @ Thunder'),
    ('Grayson Allen', 'Suns @ Thunder'),
    ('Cade Cunningham', 'Magic @ Pistons'),
    ('Paolo Banchero', 'Magic @ Pistons'),
    ('Franz Wagner', 'Magic @ Pistons'),
]

# Pull recent stats and compute predictions
pred_data = []
for player_name, game in expanded_players:
    try:
        matches = players.find_players_by_full_name(player_name)
        if not matches:
            continue
        pid = matches[0]['id']
        log = playergamelog.PlayerGameLog(player_id=pid, season='2025-26')
        df = log.get_data_frames()[0]
        if len(df) < 5:
            continue
        last_20 = df.head(20) if len(df) >= 20 else df
        pred_data.append({
            'player_name': player_name,
            'game': game,
            'pts_pred': round(last_20['PTS'].mean(), 1),
            'reb_pred': round(last_20['REB'].mean(), 1),
            'ast_pred': round(last_20['AST'].mean(), 1)
        })
        time.sleep(0.4)
    except:
        pass

# Hardcoded consensus lines from your Session 6 pull
# (In a real system these would come from live API calls)
consensus_lines = [
    # Points
    {'player_name': 'LeBron James', 'market': 'points', 'consensus_line': 25.5},
    {'player_name': 'Alperen Sengun', 'market': 'points', 'consensus_line': 21.5},
    {'player_name': 'Amen Thompson', 'market': 'points', 'consensus_line': 20.5},
    {'player_name': 'Jabari Smith Jr', 'market': 'points', 'consensus_line': 16.5},
    {'player_name': 'Deandre Ayton', 'market': 'points', 'consensus_line': 12.5},
    {'player_name': 'Luke Kennard', 'market': 'points', 'consensus_line': 12.5},
    {'player_name': 'Rui Hachimura', 'market': 'points', 'consensus_line': 14.0},
    {'player_name': 'Tyrese Maxey', 'market': 'points', 'consensus_line': 25.5},
    {'player_name': 'Jaylen Brown', 'market': 'points', 'consensus_line': 24.5},
    {'player_name': 'Jayson Tatum', 'market': 'points', 'consensus_line': 23.5},
    {'player_name': 'Nikola Vucevic', 'market': 'points', 'consensus_line': 8.5},
    {'player_name': 'Kelly Oubre Jr', 'market': 'points', 'consensus_line': 13.5},
    {'player_name': 'Shai Gilgeous-Alexander', 'market': 'points', 'consensus_line': 30.5},
    {'player_name': 'Devin Booker', 'market': 'points', 'consensus_line': 23.5},
    {'player_name': 'Jalen Green', 'market': 'points', 'consensus_line': 18.5},
    {'player_name': 'Jalen Williams', 'market': 'points', 'consensus_line': 17.5},
    {'player_name': 'Chet Holmgren', 'market': 'points', 'consensus_line': 15.5},
    {'player_name': 'Grayson Allen', 'market': 'points', 'consensus_line': 9.5},
    {'player_name': 'Cade Cunningham', 'market': 'points', 'consensus_line': 26.5},
    {'player_name': 'Paolo Banchero', 'market': 'points', 'consensus_line': 22.5},
    {'player_name': 'Franz Wagner', 'market': 'points', 'consensus_line': 17.5},
]

# Build the picks DataFrame
preds_df = pd.DataFrame(pred_data)
lines_df = pd.DataFrame(consensus_lines)

# Only points market for simplicity
merged = preds_df.merge(lines_df, on='player_name', how='inner')
merged['our_prediction'] = merged['pts_pred']
merged['diff_vs_line'] = merged['our_prediction'] - merged['consensus_line']
merged['lean'] = merged['diff_vs_line'].apply(
    lambda x: 'OVER' if x > 0 else 'UNDER'
)
merged['abs_diff'] = merged['diff_vs_line'].abs()
merged['date'] = datetime.now().strftime('%Y-%m-%d')
merged['actual_value'] = None
merged['result'] = None

# Save for the dashboard
output = merged[['player_name', 'game', 'market', 'our_prediction',
                 'consensus_line', 'diff_vs_line', 'lean', 'abs_diff',
                 'date', 'actual_value', 'result']]
output.to_csv('multi_market_picks.csv', index=False)

print(f"Saved {len(output)} picks to multi_market_picks.csv")
print(output.sort_values('abs_diff', ascending=False).head(10).to_string(index=False))

Saved 21 picks to multi_market_picks.csv
    player_name             game market  our_prediction  consensus_line  diff_vs_line  lean  abs_diff       date actual_value result
  Grayson Allen   Suns @ Thunder points            16.3             9.5           6.8  OVER       6.8 2026-04-19         None   None
Cade Cunningham  Magic @ Pistons points            20.6            26.5          -5.9 UNDER       5.9 2026-04-19         None   None
   LeBron James Rockets @ Lakers points            19.8            25.5          -5.7 UNDER       5.7 2026-04-19         None   None
   Devin Booker   Suns @ Thunder points            29.0            23.5           5.5  OVER       5.5 2026-04-19         None   None
   Luke Kennard Rockets @ Lakers points             8.4            12.5          -4.1 UNDER       4.1 2026-04-19         None   None
  Rui Hachimura Rockets @ Lakers points            11.0            14.0          -3.0 UNDER       3.0 2026-04-19         None   None
   Jaylen Brown Sixers @ Cel

In [20]:
import os
import pandas as pd

# Check if file exists
if os.path.exists('multi_market_picks.csv'):
    df = pd.read_csv('multi_market_picks.csv')
    print(f"✓ File exists with {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}")
    print("\nFirst 5 rows:")
    print(df.head())
else:
    print("✗ File does NOT exist — the pick generation cell didn't run or failed")

✓ File exists with 21 rows
Columns: ['player_name', 'game', 'market', 'our_prediction', 'consensus_line', 'diff_vs_line', 'lean', 'abs_diff', 'date', 'actual_value', 'result']

First 5 rows:
       player_name              game  market  our_prediction  consensus_line  \
0     LeBron James  Rockets @ Lakers  points            19.8            25.5   
1   Alperen Sengun  Rockets @ Lakers  points            20.7            21.5   
2    Amen Thompson  Rockets @ Lakers  points            20.8            20.5   
3  Jabari Smith Jr  Rockets @ Lakers  points            16.6            16.5   
4    Rui Hachimura  Rockets @ Lakers  points            11.0            14.0   

   diff_vs_line   lean  abs_diff        date  actual_value  result  
0          -5.7  UNDER       5.7  2026-04-19           NaN     NaN  
1          -0.8  UNDER       0.8  2026-04-19           NaN     NaN  
2           0.3   OVER       0.3  2026-04-19           NaN     NaN  
3           0.1   OVER       0.1  2026-04-19        

In [23]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import os
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime

st.set_page_config(page_title="SportsBook Edge", page_icon="🏀", layout="wide", initial_sidebar_state="expanded")

st.markdown("""
<style>
    .main { padding-top: 1rem; }
    [data-testid="stMetric"] { background-color: #1a1d23; padding: 15px; border-radius: 8px; border: 1px solid #2d3139; }
    [data-testid="stMetricLabel"] { color: #8b92a0; font-size: 0.8rem; }
    [data-testid="stMetricValue"] { color: #e8eaed; font-size: 1.8rem; }
    h1 { color: #e8eaed; }
    h2, h3 { color: #c9cdd4; }
</style>
""", unsafe_allow_html=True)

# NO CACHING on picks so we always read fresh
def load_games():
    conn = sqlite3.connect('sportsbook.db')
    df = pd.read_sql("SELECT * FROM nba_player_games ORDER BY player_id, game_date", conn)
    df['game_date'] = pd.to_datetime(df['game_date'])
    for col in ['points', 'rebounds', 'assists', 'minutes', 'fgm', 'fga', 'fg3m', 'fg3a']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    conn.close()
    return df

def load_picks():
    # Diagnostic version - no caching, report what we find
    cwd = os.getcwd()
    files_in_dir = os.listdir(cwd)

    if 'multi_market_picks.csv' in files_in_dir:
        df = pd.read_csv('multi_market_picks.csv')
        return df, f"FOUND in {cwd}", files_in_dir
    else:
        return pd.DataFrame(), f"NOT FOUND in {cwd}", files_in_dir

games_df = load_games()
picks_df, status_msg, files_in_dir = load_picks()

st.sidebar.title("🏀 SportsBook Edge")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigate", ["Overview", "Today's Picks", "Player Explorer", "Performance Tracker"])
st.sidebar.markdown("---")
st.sidebar.caption(f"Data as of {datetime.now().strftime('%b %d, %Y')}")
st.sidebar.caption(f"{len(games_df)} games | {games_df['player_name'].nunique()} players")

if page == "Overview":
    st.title("📊 SportsBook Edge Dashboard")
    st.markdown("Personal NBA prop betting analytics")
    st.markdown("---")
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Total Picks Tracked", len(picks_df))
    with col2:
        st.metric("Players in Database", games_df['player_name'].nunique())
    with col3:
        st.metric("Games Analyzed", len(games_df))
    with col4:
        if len(picks_df) > 0 and 'market' in picks_df.columns:
            st.metric("Markets Covered", picks_df['market'].nunique())
        else:
            st.metric("Markets Covered", "—")

    st.markdown("---")
    st.subheader("Points Distribution")
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=games_df['points'].dropna(), nbinsx=40, marker_color='#4a9eff', opacity=0.8))
    fig.update_layout(template='plotly_dark', height=400, xaxis_title="Points Scored", yaxis_title="Frequency", showlegend=False)
    st.plotly_chart(fig, use_container_width=True)

elif page == "Today's Picks":
    st.title("🎯 Current Picks")

    # DIAGNOSTIC OUTPUT
    st.info(f"**Diagnostic:** File status: {status_msg}")
    st.info(f"**Picks rows loaded:** {len(picks_df)}")
    with st.expander("Files in working directory"):
        st.write(files_in_dir)

    if len(picks_df) == 0:
        st.warning("No picks loaded — check diagnostic above to see what's happening")
    else:
        st.success(f"✓ {len(picks_df)} picks loaded successfully")
        col1, col2 = st.columns(2)
        with col1:
            markets = ['All'] + sorted(picks_df['market'].unique().tolist())
            selected_market = st.selectbox("Market", markets)
        with col2:
            leans = ['All'] + sorted(picks_df['lean'].unique().tolist())
            selected_lean = st.selectbox("Lean", leans)

        filtered = picks_df.copy()
        if selected_market != 'All':
            filtered = filtered[filtered['market'] == selected_market]
        if selected_lean != 'All':
            filtered = filtered[filtered['lean'] == selected_lean]

        display_cols = [c for c in ['player_name', 'game', 'market', 'our_prediction', 'consensus_line', 'diff_vs_line', 'lean'] if c in filtered.columns]
        st.dataframe(filtered[display_cols], use_container_width=True, hide_index=True)

elif page == "Player Explorer":
    st.title("👤 Player Explorer")
    player_list = sorted(games_df['player_name'].unique())
    selected_player = st.selectbox("Select a player", player_list)
    player_df = games_df[games_df['player_name'] == selected_player].sort_values('game_date')
    recent = player_df.tail(20)
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Last 20 Points", f"{recent['points'].mean():.1f}")
    with col2:
        st.metric("Last 20 Rebounds", f"{recent['rebounds'].mean():.1f}")
    with col3:
        st.metric("Last 20 Assists", f"{recent['assists'].mean():.1f}")
    with col4:
        st.metric("Last 20 Minutes", f"{recent['minutes'].mean():.1f}")

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=player_df['game_date'], y=player_df['points'], mode='lines+markers', name='Points', line=dict(color='#4a9eff', width=2)))
    player_df['rolling_10'] = player_df['points'].rolling(10, min_periods=3).mean()
    fig.add_trace(go.Scatter(x=player_df['game_date'], y=player_df['rolling_10'], mode='lines', name='10-Game Avg', line=dict(color='#f59e0b', width=3, dash='dot')))
    fig.update_layout(template='plotly_dark', height=400)
    st.plotly_chart(fig, use_container_width=True)

elif page == "Performance Tracker":
    st.title("📈 Paper-Trade Performance")
    st.info(f"Tracking {len(picks_df)} picks")
    if len(picks_df) > 0:
        st.dataframe(picks_df.head(20), use_container_width=True, hide_index=True)

st.markdown("---")
st.caption("Built with Streamlit | Data from NBA Stats API and The Odds API")

Overwriting app.py


In [26]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.graph_objects as go
from datetime import datetime

st.set_page_config(page_title="SportsBook Edge", page_icon="🏀", layout="wide", initial_sidebar_state="expanded")

st.markdown("""
<style>
    .main { padding-top: 1rem; }
    [data-testid="stMetric"] { background-color: #1a1d23; padding: 15px; border-radius: 8px; border: 1px solid #2d3139; }
    [data-testid="stMetricLabel"] { color: #8b92a0; font-size: 0.8rem; }
    [data-testid="stMetricValue"] { color: #e8eaed; font-size: 1.8rem; }
    h1 { color: #e8eaed; }
    h2, h3 { color: #c9cdd4; }
</style>
""", unsafe_allow_html=True)

def load_games():
    conn = sqlite3.connect('sportsbook.db')
    df = pd.read_sql("SELECT * FROM nba_player_games ORDER BY player_id, game_date", conn)
    df['game_date'] = pd.to_datetime(df['game_date'])
    for col in ['points', 'rebounds', 'assists', 'minutes', 'fgm', 'fga', 'fg3m', 'fg3a']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    conn.close()
    return df

def load_picks():
    try:
        return pd.read_csv('multi_market_picks.csv')
    except FileNotFoundError:
        return pd.DataFrame()

games_df = load_games()
picks_df = load_picks()

st.sidebar.title("🏀 SportsBook Edge")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigate", ["Overview", "Today's Picks", "Player Explorer", "Performance Tracker"])
st.sidebar.markdown("---")
st.sidebar.caption(f"Data as of {datetime.now().strftime('%b %d, %Y')}")
st.sidebar.caption(f"{len(games_df)} games | {games_df['player_name'].nunique()} players")

if page == "Overview":
    st.title("📊 SportsBook Edge Dashboard")
    st.markdown("Personal NBA prop betting analytics — model vs. market")
    st.markdown("---")
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Total Picks Tracked", len(picks_df))
    with col2:
        st.metric("Players in Database", games_df['player_name'].nunique())
    with col3:
        st.metric("Games Analyzed", len(games_df))
    with col4:
        st.metric("Markets Covered", picks_df['market'].nunique() if len(picks_df) and 'market' in picks_df else "—")
    st.markdown("---")
    st.subheader("Points Distribution")
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=games_df['points'].dropna(), nbinsx=40, marker_color='#4a9eff', opacity=0.8))
    fig.update_layout(template='plotly_dark', height=400, xaxis_title="Points Scored", yaxis_title="Frequency", showlegend=False)
    st.plotly_chart(fig, use_container_width=True)
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Top 10 Scoring Averages")
        tops = games_df.groupby('player_name')['points'].mean().sort_values(ascending=False).head(10).round(1).reset_index()
        tops.columns = ['Player', 'Avg Points']
        st.dataframe(tops, use_container_width=True, hide_index=True)
    with col2:
        st.subheader("Top 10 Rebounders")
        topr = games_df.groupby('player_name')['rebounds'].mean().sort_values(ascending=False).head(10).round(1).reset_index()
        topr.columns = ['Player', 'Avg Rebounds']
        st.dataframe(topr, use_container_width=True, hide_index=True)

elif page == "Today's Picks":
    st.title("🎯 Current Picks")
    if len(picks_df) == 0:
        st.warning("No picks tracked yet.")
    else:
        st.markdown(f"**{len(picks_df)} picks in tracking log**")
        col1, col2 = st.columns(2)
        with col1:
            markets = ['All'] + sorted(picks_df['market'].unique().tolist())
            selected_market = st.selectbox("Market", markets)
        with col2:
            leans = ['All'] + sorted(picks_df['lean'].unique().tolist())
            selected_lean = st.selectbox("Lean", leans)
        filtered = picks_df.copy()
        if selected_market != 'All':
            filtered = filtered[filtered['market'] == selected_market]
        if selected_lean != 'All':
            filtered = filtered[filtered['lean'] == selected_lean]
        display_cols = [c for c in ['player_name', 'game', 'market', 'our_prediction', 'consensus_line', 'diff_vs_line', 'lean'] if c in filtered.columns]
        st.dataframe(filtered[display_cols].sort_values('diff_vs_line', key=abs, ascending=False), use_container_width=True, hide_index=True)

elif page == "Player Explorer":
    st.title("👤 Player Explorer")
    player_list = sorted(games_df['player_name'].unique())
    selected_player = st.selectbox("Select a player", player_list)
    player_df = games_df[games_df['player_name'] == selected_player].sort_values('game_date')
    recent = player_df.tail(20)
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Last 20 Points", f"{recent['points'].mean():.1f}")
    with col2:
        st.metric("Last 20 Rebounds", f"{recent['rebounds'].mean():.1f}")
    with col3:
        st.metric("Last 20 Assists", f"{recent['assists'].mean():.1f}")
    with col4:
        st.metric("Last 20 Minutes", f"{recent['minutes'].mean():.1f}")
    st.markdown("---")
    st.subheader(f"{selected_player} — Points by Game")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=player_df['game_date'], y=player_df['points'], mode='lines+markers', name='Points', line=dict(color='#4a9eff', width=2)))
    player_df['rolling_10'] = player_df['points'].rolling(10, min_periods=3).mean()
    fig.add_trace(go.Scatter(x=player_df['game_date'], y=player_df['rolling_10'], mode='lines', name='10-Game Avg', line=dict(color='#f59e0b', width=3, dash='dot')))
    fig.update_layout(template='plotly_dark', height=400, hovermode='x unified')
    st.plotly_chart(fig, use_container_width=True)
    st.subheader("Last 15 Games")
    rd = player_df.tail(15)[['game_date', 'matchup', 'minutes', 'points', 'rebounds', 'assists', 'plus_minus']].sort_values('game_date', ascending=False)
    rd['game_date'] = rd['game_date'].dt.strftime('%b %d, %Y')
    st.dataframe(rd, use_container_width=True, hide_index=True)

elif page == "Performance Tracker":
    st.title("📈 Paper-Trade Performance")
    if len(picks_df) == 0:
        st.warning("No picks in tracking log.")
    elif 'result' not in picks_df.columns or picks_df['result'].notna().sum() == 0:
        st.info("Fill in actual_value and result columns in multi_market_picks.csv after games complete, then reload.")
        st.markdown("**Preview of picks awaiting results:**")
        st.dataframe(picks_df.head(25), use_container_width=True, hide_index=True)
    else:
        completed = picks_df.dropna(subset=['result']).copy()
        total = len(completed)
        wins = (completed['result'] == 'WIN').sum()
        losses = (completed['result'] == 'LOSS').sum()
        hit_rate = wins / (wins + losses) if (wins + losses) > 0 else 0
        col1, col2, col3, col4 = st.columns(4)
        with col1:
            st.metric("Completed", total)
        with col2:
            st.metric("Wins", wins)
        with col3:
            st.metric("Losses", losses)
        with col4:
            st.metric("Hit Rate", f"{hit_rate:.1%}")

st.markdown("---")
st.caption("Built with Streamlit | Data from NBA Stats API and The Odds API")

Overwriting app.py
